In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: פונקציית Qiskit מאת ColibriTD
> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan ו-On-Prem (דרך IBM Quantum Platform API) Plan. הן נמצאות בסטטוס גרסת תצוגה מקדימה וכפופות לשינויים.
## סקירה כללית
פותר המשוואות הדיפרנציאליות החלקיות (PDE) המוצג כאן הוא חלק מפלטפורמת Quantum Innovative Computing Kit (QUICK) שלנו (QUICK-PDE), ומאורז כפונקציית Qiskit. עם פונקציית QUICK-PDE, אתה יכול לפתור משוואות דיפרנציאליות חלקיות ספציפיות לתחום על QPU של IBM Quantum. פונקציה זו מבוססת על האלגוריתם המתואר ב[מאמר תיאור H-DES של ColibriTD.](https://arxiv.org/abs/2410.01130) אלגוריתם זה יכול לפתור בעיות פיזיקה מרובות מורכבות, החל מדינמיקת נוזלים חישובית (CFD) ועיוות חומרים (MD), ועוד שימושים נוספים בקרוב.

כדי להתמודד עם המשוואות הדיפרנציאליות, פתרונות הניסיון מקודדים כצירופים לינאריים של פונקציות אורתוגונליות (בדרך כלל פולינומי Chebyshev, וספציפית יותר $2^n$ מהם כאשר $n$ הוא מספר ה-Qubit המקודדים את הפונקציה שלך), שמופרמטרים על ידי זוויות של מעגל קוונטי משתנה (VQC). ה-ansatz מייצר מצב המקודד את הפונקציה, אשר מוערך על ידי אובזרבבלים שצירופיהם מאפשרים הערכת הפונקציה בכל הנקודות. לאחר מכן אתה יכול להעריך את פונקציית ההפסד שבה מקודדות המשוואות הדיפרנציאליות, ולכוונן את הזוויות בלולאה היברידית, כפי שמוצג בהמשך. פתרונות הניסיון מתקרבים בהדרגה לפתרונות האמיתיים עד שמגיעים לתוצאה משביעת רצון.

![תרשים זרימה של פונקציית QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

בנוסף ללולאה ההיברידית הזו, אתה יכול גם לשרשר מייעלים שונים. זה שימושי כאשר אתה רוצה שמייעל גלובלי ימצא קבוצה טובה של זוויות, ולאחר מכן מייעל מכוון יותר יעקוב אחרי גרדיאנט אל קבוצת הזוויות השכנות הטובה ביותר. במקרה של דינמיקת נוזלים חישובית (CFD), רצף האופטימיזציה ברירת המחדל מפיק את התוצאות הטובות ביותר - אך במקרה של עיוות חומרים (MD), בעוד שברירת המחדל מספקת תוצאות טובות, אתה יכול להגדיר אותה עוד יותר לצרכים ספציפיים לבעיה.

שים לב שעבור כל משתנה של הפונקציה, אנו מציינים את מספר ה-Qubit (איתו אתה יכול להתנסות). על ידי שכבוב 10 Circuit זהים והערכת 10 אובזרבבלים זהים על Qubit שונים לאורך Circuit גדול אחד, אתה יכול להפחית רעש בתוך תהליך האופטימיזציה של CMA, תוך הסתמכות על שיטת לומד הרעש, ולהפחית משמעותית את מספר ה-shots הנדרשים.
## קלט/פלט
### דינמיקת נוזלים חישובית
משוואת Burgers ללא צמיגות מדגמת זרימת נוזלים ללא צמיגות כדלקמן:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ מייצגת את שדה מהירות הנוזל. לשימוש זה יש תנאי גבול זמני: אתה יכול לבחור את תנאי ההתחלה ולאחר מכן לאפשר למערכת להתרגע. נכון לעכשיו, תנאי ההתחלה המקובלים היחידים הם פונקציות לינאריות: $ax + b$.

הארגומנטים עבור משוואות הדיפרנציאל של CFD נמצאים על רשת קבועה, כדלקמן:

- $t$ נמצא בין 0 ל-0.95 עם 30 נקודות דגימה. $x$ נמצא בין 0 ל-0.95 עם גודל צעד של 0.2375.

### עיוות חומרים
שימוש זה מתמקד בעיוות היפו-אלסטי עם מבחן המתיחה החד-ממדי, שבו מוט הקבוע במרחב נמשך בקצהו השני. אנו מתארים את הבעיה כדלקמן:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ מייצג את מודול הנפח של החומר הנמתח, $n$ את המעריך של חוק הכוח, $b$ את הכוח ליחידת מסה, $\epsilon_0$ את גבול מתח הפרופורציה, $\sigma_0$ את גבול עיוות הפרופורציה, $u$ את פונקציית המתח, ו-$\sigma$ את פונקציית העיוות.

המוט הנחשב הוא באורך יחידה. לשימוש זה יש תנאי גבול למתח פני השטח $t$, או כמות העבודה הנדרשת למתוח את המוט.

הארגומנטים עבור משוואות הדיפרנציאל של MD נמצאים על רשת קבועה, כדלקמן:

- $x$ נמצא בין 0 ל-1 עם גודל צעד של 0.04.
### קלט
כדי להריץ את פונקציית ה-Qiskit QUICK-PDE, אתה יכול לכוונן את הפרמטרים הבאים:

| שם              | סוג                                                 | תיאור                                                                                                                                                                                                                                                                     | ספציפי לשימוש | דוגמה                  |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------- | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | מיתוג לבחירת מערכת המשוואות הדיפרנציאליות לפתרון                                                                                                                                                                                                                                  | לא                | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | פרמטרים של המשוואה הדיפרנציאלית (ראה את הטבלה הבאה לפרטים נוספים)                                                                                                                                                                                                              | לא                | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | מספר ה-Qubit לפי פונקציה ולפי משתנה. הפונקציה בוחרת ערכים אופטימליים, אך אם ברצונך לנסות למצוא שילוב טוב יותר, אתה יכול לעקוף את ערכי ברירת המחדל                                                                                         | לא                | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | עומק ה-ansatz לפי פונקציה. הפונקציה בוחרת ערכים אופטימליים, אך אם ברצונך לנסות למצוא שילוב טוב יותר, אתה יכול לעקוף את ערכי ברירת המחדל                                                                                           | לא                | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | מייעלים לשימוש, "CMAES" מ[ספריית ה-python `cma`](https://github.com/CMA-ES/pycma) או אחד מ[מייעלי scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)                                                                   | MD                | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | מספר ה-shots המשמשים להרצת כל Circuit. מכיוון שנדרשים מספר שלבי אופטימיזציה, אורך הרשימה חייב להיות שווה למספר המייעלים המשמשים (4 במקרה של CFD). ברירת המחדל היא `[50_000] * nb_optimizers` עבור MD ו-`[5_00, 2_000, 5_000, 10_000]` עבור CFD | לא                | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | אפשרויות להעברה למייעל. פרטי קלט זה תלויים במייעל המשמש; לפרטים ספציפיים, עיין בתיעוד המייעל המשמש                                                                                              | MD                | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | האם להתחיל עם זוויות אקראיות או זוויות שנבחרו בחוכמה. שים לב שזוויות שנבחרו בחוכמה עשויות לא לעבוד ב-100% מהמקרים. ברירת המחדל היא `"PHYSICALLY_INFORMED"`.                                                                                                         | לא                | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | שם ה-Backend לשימוש.                                                                                                                                                                                                              | לא                | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | מצב הביצוע לשימוש. ברירת המחדל היא `"job"`.                                                                                                                                                                                                                                       | לא                | `"job"`                  |

פרמטרי המשוואה הדיפרנציאלית (פרמטרים פיזיקליים ותנאי גבול) צריכים לעקוב אחר הפורמט הנתון:

| שימוש | מפתח         | סוג ערך | תיאור                              | דוגמה |
| -------- | ----------- | ---------- | ---------------------------------------- | ------- |
| CFD      | `a`         | `float`    | מקדם של ערכי ההתחלה של $u$ | `1.0`   |
| CFD      | `b`         | `float`    | היסט של ערכי ההתחלה של $u$      | `1.0`   |
| MD       | `t`         | `float`    | מתח פני שטח                           | `12.0`  |
| MD       | `K`         | `float`    | מודול נפח                             | `100.0` |
| MD       | `n`         | `int`      | חוק כוח                                | `4.0`   |
| MD       | `b`         | `float`    | כוח ליחידת מסה                      | `10.0`  |
| MD       | `epsilon_0` | `float`    | גבול מתח פרופורציה                | `0.1`   |
| MD       | `sigma_0`   | `float`    | גבול עיוות פרופורציה                | `5.0`   |
### פלט
הפלט הוא מילון עם רשימת נקודות הדגימה, כמו גם ערכי הפונקציות בכל אחת מהנקודות הללו:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

צורת מערך פתרון תלויה בדגימות המשתנים:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

המיפוי בין נקודות הדגימה של משתני הפונקציה לבין ממד מערך הפתרון נעשה בסדר אלפאנומרי של שם המשתנה. לדוגמה, אם המשתנים הם `"t"` ו-`"x"`, שורה של `solution["functions"]["u"]` מייצגת את ערכי הפתרון עבור `"t"` קבוע, ועמודה של `solution["functions"]["u"]` מייצגת את ערכי הפתרון עבור `"x"` קבוע.

להלן דוגמה לאופן קבלת ערך הפונקציה עבור קבוצת קואורדינטות ספציפית:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## מדדים

הטבלה הבאה מציגה סטטיסטיקות על ריצות שונות של הפונקציה שלנו.

| דוגמה                      | מספר Qubit | אתחול        | שגיאה     | זמן כולל (דק') | שימוש בסביבת הריצה (דק') |
| ---------------------------- | ---------------- | --------------------- | --------- | ---------------- | ------------------- |
| משוואת Burgers ללא צמיגות   | 50               | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66               | 25                  |
| מבחן מתיחה היפו-אלסטי חד-ממדי  | 18               | `RANDOM`              | $10^{-2}$ | 123              | 100                 |

## תחילת העבודה

מלא את [הטופס לבקשת גישה לפונקציית QUICK-PDE.](https://forms.cloud.microsoft/e/3Wi9cbjQPK) לאחר מכן, בהנחה שכבר [שמרת את חשבונך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך, בחר את הפונקציה כדלקמן:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## דוגמאות

כדי להתחיל, נסה אחת מהדוגמאות הבאות:

### דינמיקת נוזלים חישובית (CFD)

כאשר תנאי ההתחלה מוגדרים ל-$u(0,x) = x$, התוצאות הן כדלקמן:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

בדוק את [הסטטוס](/guides/functions#check-job-status) של עומס העבודה של פונקציית ה-Qiskit שלך או החזר [תוצאות](/guides/functions#retrieve-results) כדלקמן:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![פלט של תא הקוד הקודם](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### עיוות חומרים
שימוש הביצוע של עיוות חומרים דורש את הפרמטרים הפיזיקליים של החומר שלך ואת הכוח המופעל, כדלקמן:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![פלט של תא הקוד הקודם](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## אחזור הודעות שגיאה

אם סטטוס עומס העבודה שלך הוא `ERROR`, השתמש ב-`job.error_message()` כדי לאחזר את הודעת השגיאה לסיוע בניפוי באגים, כדלקמן: